In [1]:
import random
import pandas as pd
from itertools import combinations

In [2]:
# Board positions in order (left to right, top to bottom)
positions = [
    'top-left', 'top-center', 'top-right',
    'middle-left', 'center', 'middle-right',
    'bottom-left', 'bottom-center', 'bottom-right'
]

# Position index mapping
position_index = {pos: i for i, pos in enumerate(positions)}

In [19]:
def positions_to_text(pos_list):
    """Convert a list of positions to natural language"""
    if len(pos_list) == 0:
        return "none"
    elif len(pos_list) == 1:
        return f"the {pos_list[0]}"
    elif len(pos_list) == 2:
        return f"the {pos_list[0]} and {pos_list[1]}"
    else:
        return ", ".join(f"the {p}" for p in pos_list[:-1]) + f", and the {pos_list[-1]}"

def create_board_state(x_positions, o_positions):
    """Create board state string"""
    board = ['_'] * 9
    for pos in x_positions:
        board[position_index[pos]] = 'X'
    for pos in o_positions:
        board[position_index[pos]] = 'O'
    return '|'.join(board)

In [20]:
# Sentence templates
templates = [
    "X is in {x_pos}, O is in {o_pos}",
    "X has played in {x_pos}, O has played in {o_pos}",
    "X occupies {x_pos}, O occupies {o_pos}",
    "X is placed in {x_pos}, O is placed in {o_pos}",
    "X marks {x_pos}, O marks {o_pos}"
]

In [21]:
def generate_sentence(template, x_positions, o_positions):
    x_pos = positions_to_text(x_positions)
    o_pos = positions_to_text(o_positions)
    return template.format(x_pos=x_pos, o_pos=o_pos)

# Generating sentences
sentences = []
sentence_id = 1

for i in range(4000):
    # Randomly decide how many X and O pieces
    x_count = random.randint(1, 5)
    if x_count == 5:
        o_count = 4
    else:
        o_count = random.choice([x_count, x_count - 1])
    
    # Randomly pick positions for X and O (no overlap)
    all_positions = random.sample(positions, x_count + o_count)
    x_positions = all_positions[:x_count]
    o_positions = all_positions[x_count:]
    
    # Create board state
    board_state = create_board_state(x_positions, o_positions)
    
    # Generate sentence
    template = random.choice(templates)
    sentence = generate_sentence(template, x_positions, o_positions)
    
    sentences.append({
        'sentence_id': sentence_id,
        'template_id': templates.index(template) + 1,
        'sentence': sentence,
        'board_state': board_state,
        'x_count': x_count,
        'x_positions': ', '.join(x_positions),
        'o_count': o_count,
        'o_positions': ', '.join(o_positions)
    })
    
    sentence_id += 1

print(f"Generated {len(sentences)} sentences")
print("\nExample sentences:")
for s in random.sample(sentences, 20):
    print(s['sentence'])

Generated 4000 sentences

Example sentences:
X occupies the bottom-right, the top-left, the top-right, and the top-center, O occupies the center, the bottom-left, the bottom-center, and the middle-left
X marks the top-right, the bottom-center, and the center, O marks the middle-right, the top-left, and the middle-left
X occupies the bottom-left, the center, and the middle-right, O occupies the bottom-center, the middle-left, and the bottom-right
X is in the middle-right and top-right, O is in the top-left and top-center
X marks the top-right, the top-left, and the bottom-left, O marks the middle-right, the middle-left, and the top-center
X is in the top-left, the top-right, the bottom-left, the middle-right, and the center, O is in the bottom-center, the bottom-right, the middle-left, and the top-center
X is in the center, O is in the bottom-center
X is placed in the bottom-right, O is placed in none
X marks the bottom-right, the bottom-left, the middle-right, and the top-left, O marks

In [22]:
# Check duplicates
df = pd.DataFrame(sentences)
duplicates = df[df.duplicated(subset=['sentence'])]
print(f"Duplicate sentences: {len(duplicates)}")

Duplicate sentences: 581


In [23]:
# Remove duplicates
df = df.drop_duplicates(subset=['sentence']).reset_index(drop=True)
df['sentence_id'] = range(1, len(df) + 1)

In [24]:
print(f"Cleaned dataset: {len(df)} sentences")
print("\nTemplate distribution:")
print(df['template_id'].value_counts().sort_index())
print("\nX count distribution:")
print(df['x_count'].value_counts().sort_index())
print(df['o_count'].value_counts().sort_index())

Cleaned dataset: 3419 sentences

Template distribution:
template_id
1    715
2    669
3    710
4    685
5    640
Name: count, dtype: int64

X count distribution:
x_count
1    282
2    763
3    810
4    796
5    768
Name: count, dtype: int64
o_count
0      45
1     611
2     793
3     810
4    1160
Name: count, dtype: int64


In [25]:
# Save
df.to_csv('sentence_dataset_tictactoe.csv', index=False)
print("\nSaved to sentence_dataset_tictactoe.csv")


Saved to sentence_dataset_tictactoe.csv
